# Train a Mask R-CNN model for your particle images

This notebook starts from a model already trained on general images, then adapts it to the particles in your labelled images. Starting from learned features usually works better than asking a model to learn everything from scratch on a small microscopy dataset.

**What you need**
- One or more ZIP files exported from TEMseg as **COCO** or **Instances**, or a COCO dataset containing images and polygon annotations.
- Ground-truth outlines that closely follow the particles. The next section of the project guide shows how to create them in TEMseg.
- A Google Colab runtime with a GPU. In Colab, choose **Runtime → Change runtime type → T4 GPU** before running the cells.

More varied, carefully labelled images usually help more than running many extra epochs. A small dataset can still be useful, but its validation results will be uncertain and it may not generalise to new samples.

Run cells from top to bottom with **Shift+Enter**. At the end, download `maskrcnn_best.pth` and load it in TEMseg. Training time depends strongly on image dimensions, particle count, and Colab hardware; let the first epoch finish before estimating the total time.

In [ ]:
!nvidia-smi -L
import torch

assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime > Change runtime type and select T4 GPU, then rerun this cell."
)
print(f"Ready. GPU: {torch.cuda.get_device_name(0)}")


# Step 1 — Upload your labelled images

Run the next cell and select one or more ZIP files. You can upload one TEMseg session export per ZIP or a ZIP containing several session exports.

The notebook reads **COCO** exports (`annotations.coco.json`) and **Instances** exports (`instances.json`). An export must include its source image and particle outlines. Statistics-only exports do not contain training labels.

After upload, the notebook checks the files and reports how many labelled images and particles it found. If that count looks wrong, stop here and check that the ZIP contains the original images and ground truths.

In [ ]:
!pip install -q torchvision

import shutil
import zipfile
from pathlib import Path
from google.colab import files

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

uploaded = files.upload()  # pick your data ZIP(s)
for name in uploaded:
    zpath = Path(name)
    target = DATA_DIR / zpath.stem
    target.mkdir(exist_ok=True)
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(target)
    # session ZIPs nested inside the big ZIP
    for inner in target.rglob("*.zip"):
        with zipfile.ZipFile(inner) as zf:
            zf.extractall(inner.parent / inner.stem)
        inner.unlink()
    zpath.unlink()

print("Unpacked into", DATA_DIR)


In [ ]:
import json
from collections import defaultdict

import cv2
import numpy as np


def polygons_from_segmentation(seg):
    if not isinstance(seg, list) or not seg:
        raise ValueError("unsupported segmentation (only polygons are supported)")
    parts = []
    for part in seg:
        if isinstance(part, dict):
            raise ValueError(
                "This annotation uses compressed masks instead of outlines. "
                "Re-export from TEMseg as COCO polygons or use an Instances export."
            )
        points = np.asarray(part, dtype=np.float64).reshape(-1, 2)
        if len(points) >= 3 and np.isfinite(points).all():
            parts.append(points)
    if not parts:
        raise ValueError("annotation contains no valid polygon outlines")
    return parts


def load_coco(json_path):
    with open(json_path) as file:
        coco = json.load(file)
    images = {image["id"]: image for image in coco.get("images", [])}
    annotations = defaultdict(list)
    for annotation in coco.get("annotations", []):
        if annotation.get("iscrowd", 0):
            continue
        try:
            annotations[annotation["image_id"]].append(
                polygons_from_segmentation(annotation["segmentation"])
            )
        except (KeyError, ValueError) as error:
            print(f"Skipped one annotation in {json_path.name}: {error}")

    root = json_path.parent
    records = []
    for image in images.values():
        filename = Path(image["file_name"])
        exact_path = root / filename
        image_path = exact_path if exact_path.is_file() else next(
            (path for path in root.rglob(filename.name) if path.is_file()), None
        )
        if image_path is not None:
            records.append({"image": image_path, "polygons": annotations.get(image["id"], []), "source": "coco"})
    return records


def load_instances(json_path):
    image_path = json_path.parent / "original_image.png"
    if not image_path.is_file():
        image_path = next(iter(json_path.parent.glob("*.png")), None)
    if image_path is None:
        return []

    with open(json_path) as file:
        instances = json.load(file)
    polygons = []
    for instance in instances:
        parts = []
        contours = [instance.get("contour", [])] + instance.get("extra_contours", [])
        for contour in contours:
            points = np.asarray(contour, dtype=np.float64).reshape(-1, 2)
            if len(points) >= 3 and np.isfinite(points).all():
                parts.append(points)
        if parts:
            polygons.append(parts)
    return [{"image": image_path, "polygons": polygons, "source": "instances"}]


def instance_masks_from_polygons(shape, polygons):
    masks = []
    for parts in polygons:
        mask = np.zeros(shape, dtype=np.uint8)
        cv2.fillPoly(mask, [part.round().astype(np.int32) for part in parts], 1)
        if mask.any():
            masks.append(mask)
    return np.asarray(masks, dtype=np.uint8).reshape(-1, *shape)


def mask_from_polygons(shape, polygons):
    masks = instance_masks_from_polygons(shape, polygons)
    return masks.max(axis=0) if len(masks) else np.zeros(shape, dtype=np.uint8)


records = []
for json_path in sorted(DATA_DIR.rglob("*.json")):
    try:
        if json_path.name == "instances.json":
            records.extend(load_instances(json_path))
        else:
            with open(json_path) as file:
                header = json.load(file)
            if "images" in header and "annotations" in header:
                records.extend(load_coco(json_path))
    except (OSError, json.JSONDecodeError, KeyError, TypeError, ValueError) as error:
        print(f"Could not read {json_path.name}: {error}")

# Prefer the app's Instances export if a COCO export contains the same image too.
samples = []
seen_images = set()
records.sort(key=lambda record: record.get("source") != "instances")
for record in records:
    if record["polygons"] and record["image"] not in seen_images:
        seen_images.add(record["image"])
        samples.append(record)

if not samples:
    raise ValueError("No labelled images found. Check that your ZIP includes images and COCO or Instances annotations.")

for folder in sorted(path for path in DATA_DIR.iterdir() if path.is_dir()):
    if not any(folder in sample["image"].parents for sample in samples):
        print(f"Note: {folder.name} contains no image with annotations and was skipped.")

n_particles = sum(len(sample["polygons"]) for sample in samples)
print(f"Found {len(samples)} labelled images with {n_particles} particle instances.")
if len(samples) < 20:
    print("This is a small dataset; validation scores may vary considerably between images.")

# Step 2 — Check the labels

Review these overlays before training. Red areas show the pixels the model will learn as particles. If an outline misses a particle, includes background, or is cut off, correct the ground truth in TEMseg and export the data again. Training cannot compensate for incorrect labels.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, min(4, len(samples)), figsize=(16, 4))
for ax, sample in zip(np.atleast_1d(axes), samples[:4]):
    img = cv2.imread(str(sample["image"]), cv2.IMREAD_GRAYSCALE)
    mask = mask_from_polygons(img.shape, sample["polygons"])
    overlay = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    overlay[mask.astype(bool)] = [255, 80, 80]
    ax.imshow(overlay)
    ax.set_title(f"{len(sample['polygons'])} particles")
    ax.axis("off")
plt.tight_layout()
plt.show()


# Step 3 — Choose training settings

The defaults are a starting point for fine-tuning a pretrained model. `EPOCHS` is the number of passes through the training images; `BATCH_SIZE` is how many images are processed together. Leave the learning rate alone unless you have a reason to tune it. If Colab reports that it ran out of GPU memory, try a batch size of 1.

A fixed portion of the images is held aside for validation. Those images are not used to update the model, so their results give a more useful estimate of performance on unseen images. With very few images, that estimate is noisy; add more labelled images if you can.

In [ ]:
import random

SEED = 42
EPOCHS = 20
BATCH_SIZE = 2
LEARNING_RATE = 0.001
VAL_FRACTION = 0.2
MIN_SCORE = 0.95  # TEMseg's Mask R-CNN confidence cutoff
VALIDATION_SCORE = 0.5

random.seed(SEED)
shuffled_samples = samples.copy()
random.shuffle(shuffled_samples)

if len(shuffled_samples) < 5:
    raise ValueError(
        f"Found only {len(shuffled_samples)} labelled images. Add at least 5 so "
        "the notebook can reserve images for validation; more varied images are strongly recommended."
    )

n_val = max(1, round(len(shuffled_samples) * VAL_FRACTION))
n_val = min(n_val, len(shuffled_samples) - 1)
val_samples = shuffled_samples[:n_val]
train_samples = shuffled_samples[n_val:]
print(f"{len(train_samples)} images for training; {len(val_samples)} held out for validation.")

# Step 4 — Train and track validation performance

The model is fine-tuned from general-purpose Mask R-CNN weights. It sees random rotations and flips of the training images, while validation images stay unchanged. This gives it useful variety without letting the validation score benefit from augmented copies of training data.

Each epoch reports the average training loss and validation precision, recall, and F1 at an IoU match threshold of 0.5. IoU measures how much a predicted mask overlaps a labelled particle. These scores are a practical check, not a guarantee: inspect the overlays in the next step and try the model on images it has never seen.

The notebook saves `maskrcnn_best.pth` whenever validation F1 improves. Training can take a while, especially for large images or crowded fields of view.

In [ ]:
import time

import torch
from torch.utils.data import DataLoader, Dataset
from torchvision.models.detection import MaskRCNN_ResNet50_FPN_Weights, maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor


class ParticleDataset(Dataset):
    def __init__(self, samples, augment):
        self.samples = samples
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img = cv2.imread(str(sample["image"]), cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise ValueError(f"Could not read image: {sample['image']}")

        masks = instance_masks_from_polygons(img.shape, sample["polygons"])
        if self.augment:
            k = random.randrange(4)
            img = np.rot90(img, k)
            masks = np.rot90(masks, k, axes=(1, 2))
            if random.random() < 0.5:
                img = np.fliplr(img)
                masks = np.flip(masks, axis=2)
            if random.random() < 0.5:
                img = np.flipud(img)
                masks = np.flip(masks, axis=1)

        img = np.ascontiguousarray(img)
        masks = np.ascontiguousarray(masks)
        image = torch.from_numpy(np.repeat(img[None, :, :], 3, axis=0)).float() / 255.0

        boxes = []
        valid_masks = []
        for mask in masks:
            ys, xs = np.nonzero(mask)
            if len(xs) == 0:
                continue
            boxes.append([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1])
            valid_masks.append(mask)

        target = {
            "boxes": torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4),
            "masks": torch.as_tensor(np.asarray(valid_masks, dtype=np.uint8)).reshape(-1, *img.shape),
            "labels": torch.ones(len(valid_masks), dtype=torch.int64),
        }
        return image, target


def collate(batch):
    return tuple(zip(*batch))


def build_model():
    model = maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT)
    box_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(box_features, 2)
    mask_features = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(mask_features, 256, 2)
    return model


def move_targets(targets, device):
    return [{key: value.to(device) for key, value in target.items()} for target in targets]


def validation_counts(model, loader, device, score_threshold):
    true_positives = 0
    false_positives = 0
    false_negatives = 0
    model.eval()

    with torch.no_grad():
        for images, targets in loader:
            predictions = model([image.to(device) for image in images])
            for prediction, target in zip(predictions, targets):
                truth_masks = target["masks"].bool().to(device)
                matched_truth = set()
                order = prediction["scores"].argsort(descending=True)
                order = order[prediction["scores"][order] >= score_threshold]

                for prediction_index in order.tolist():
                    predicted_mask = prediction["masks"][prediction_index, 0] > 0.5
                    if len(truth_masks) == 0:
                        false_positives += 1
                        continue
                    intersections = (truth_masks & predicted_mask).flatten(1).sum(dim=1).float()
                    unions = (truth_masks | predicted_mask).flatten(1).sum(dim=1).float()
                    ious = intersections / unions.clamp(min=1)
                    for truth_index in matched_truth:
                        ious[truth_index] = 0
                    best_iou, best_index = ious.max(dim=0)
                    if best_iou.item() >= 0.5:
                        true_positives += 1
                        matched_truth.add(int(best_index.item()))
                    else:
                        false_positives += 1
                false_negatives += len(truth_masks) - len(matched_truth)

    precision = true_positives / max(true_positives + false_positives, 1)
    recall = true_positives / max(true_positives + false_negatives, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return precision, recall, f1


def mask_from_polygons(shape, polygons):
    masks = instance_masks_from_polygons(shape, polygons)
    return masks.max(axis=0) if len(masks) else np.zeros(shape, dtype=np.uint8)


device = torch.device("cuda")
model = build_model().to(device)
optimizer = torch.optim.SGD(
    [parameter for parameter in model.parameters() if parameter.requires_grad],
    lr=LEARNING_RATE,
    momentum=0.9,
    weight_decay=0.0005,
)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=max(EPOCHS // 2, 1), gamma=0.1)

train_loader = DataLoader(
    ParticleDataset(train_samples, augment=True),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate,
)
val_loader = DataLoader(
    ParticleDataset(val_samples, augment=False),
    batch_size=1,
    shuffle=False,
    collate_fn=collate,
)

best_f1 = -1.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    started = time.time()
    total_loss = 0.0

    for batch_index, (images, targets) in enumerate(train_loader, start=1):
        images = [image.to(device) for image in images]
        targets = move_targets(targets, device)
        losses = model(images, targets)
        loss = sum(losses.values())
        if not torch.isfinite(loss):
            raise RuntimeError(f"Training loss became non-finite in epoch {epoch}, batch {batch_index}.")
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        print(f"\rEpoch {epoch}/{EPOCHS} — batch {batch_index}/{len(train_loader)} — loss {loss.item():.3f}", end="")

    train_loss = total_loss / len(train_loader)
    precision, recall, val_f1 = validation_counts(model, val_loader, device, VALIDATION_SCORE)
    scheduler.step()

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), "maskrcnn_best.pth")

    print(
        f"\rEpoch {epoch}/{EPOCHS} — train loss {train_loss:.3f} — "
        f"validation precision {precision:.3f}, recall {recall:.3f}, F1 {val_f1:.3f} "
        f"— {time.time() - started:.0f}s"
    )

print(f"Best validation F1: {best_f1:.3f}. Saved weights to maskrcnn_best.pth.")

# Step 5 — Review predictions

The plots below use held-out validation images, which were not used to train the model. Red shows predicted particle masks; the green outline shows the ground truth. Look for missed particles, extra detections, and outlines that are consistently too large or too small.

The preview uses TEMseg's confidence cutoff of 0.95. A model can have a reasonable validation F1 at the lower diagnostic threshold while still producing few detections at 0.95. If that happens, inspect the labels and add representative images before increasing training time.

In [ ]:
import matplotlib.pyplot as plt

model = build_model().to(device)
model.load_state_dict(torch.load("maskrcnn_best.pth", map_location=device, weights_only=True))
model.eval()
print("Best-validation weights loaded. These use the same model structure as TEMseg.")

preview_samples = val_samples[:4]
fig, axes = plt.subplots(1, len(preview_samples), figsize=(4 * len(preview_samples), 4))
for ax, sample in zip(np.atleast_1d(axes), preview_samples):
    img = cv2.imread(str(sample["image"]), cv2.IMREAD_GRAYSCALE)
    image = torch.from_numpy(np.repeat(img[None, :, :], 3, axis=0)).float() / 255.0
    with torch.no_grad():
        prediction = model([image.to(device)])[0]
    keep = prediction["scores"] >= MIN_SCORE
    canvas = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    for mask in prediction["masks"][keep, 0].cpu().numpy():
        canvas[mask > 0.5] = [255, 80, 80]
    truth = mask_from_polygons(img.shape, sample["polygons"])
    contours, _ = cv2.findContours(truth, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(canvas, contours, -1, (80, 255, 80), 1)
    ax.imshow(canvas)
    ax.set_title(f"{sample['image'].name}\n{int(keep.sum())} predictions at 0.95")
    ax.axis("off")
plt.tight_layout()
plt.show()

# Step 6 — Download your weights

Run the cell and save the `maskrcnn_best.pth` file somewhere you can find it —
usually your Downloads folder.


In [ ]:
from google.colab import files

files.download("maskrcnn_best.pth")


# Step 7 — Use your model in TEMseg

1. Rename the downloaded file to **`maskrcnn_best_model.pth`**.
2. Put it in TEMseg's weights folder:
   - **Mac:** `~/Library/Application Support/TEMseg/weights/`
   - **Windows:** `%LOCALAPPDATA%\TEMseg\weights\`
   - **Linux:** `~/.local/share/TEMseg/weights/`
3. Restart TEMseg and select the Mask R-CNN model.

You can also keep the built-in weights and point `TEMSEG_WEIGHTS_DIR` to a folder containing your custom weights, if you already use that setting.

TEMseg currently displays Mask R-CNN detections with confidence above **0.95**. The preview uses the same cutoff. If the model misses particles, first check that the validation overlays match the ground truth and that your training images resemble the images you plan to analyze. Add varied, accurately labelled images before simply increasing the number of epochs.